# Export TensorFlow.js Model

這個 notebook 展示如何將已訓練好的 `best.pt` 匯出成 TensorFlow SavedModel，然後轉換為 TensorFlow.js Graph Model。

## Step 1: 確認 best.pt 是否存在

In [ ]:
from pathlib import Path

best_pt = Path('../models/best.pt')
print('best.pt path:', best_pt.resolve())
print('Exists:', best_pt.exists())
assert best_pt.exists(), 'best.pt not found. Please put best.pt in ai/models/'

## Step 2: 使用 Ultralytics 將 best.pt 匯出為 SavedModel

In [ ]:
from pathlib import Path
from ultralytics import YOLO

best_pt = Path('../models/best.pt')
saved_model_dir = Path('../exports/saved_model').resolve()
saved_model_dir.mkdir(parents=True, exist_ok=True)

print('Exporting SavedModel to:', saved_model_dir)
model = YOLO(str(best_pt))
model.export(format='saved_model', save_dir=str(saved_model_dir))

## Step 3: 將 SavedModel 轉換為 TensorFlow.js Graph Model

In [ ]:
import subprocess
from pathlib import Path

saved_model_dir = Path('../exports/saved_model')
tfjs_model_dir = Path('../exports/tfjs_model')
tfjs_model_dir.mkdir(parents=True, exist_ok=True)

command = [
    'tensorflowjs_converter',
    '--input_format=tf_saved_model',
    '--output_format=tfjs_graph_model',
    str(saved_model_dir),
    str(tfjs_model_dir),
]
print('Running:', ' '.join(command))
subprocess.run(command, check=True)

## Step 4: 驗證模型檔案是否存在

In [ ]:
from pathlib import Path

tfjs_model_dir = Path('../exports/tfjs_model')
model_json = tfjs_model_dir / 'model.json'
bin_files = list(tfjs_model_dir.glob('*.bin'))

print('TensorFlow.js model directory:', tfjs_model_dir.resolve())
print('model.json exists:', model_json.exists())
print('bin files found:', [p.name for p in bin_files])
assert model_json.exists(), 'model.json not found in exports/tfjs_model/'
assert bin_files, 'No .bin files found in exports/tfjs_model/'

## Step 5: TensorFlow.js 載入方式與 React 路徑

- 安裝 TensorFlow.js：`npm install @tensorflow/tfjs`
- 在 React 中載入 Graph Model：

```ts
import * as tf from '@tensorflow/tfjs'

const model = await tf.loadGraphModel('/models/model.json')
```

- 如果模型檔案放在 React `public/models/` 資料夾中，`tf.loadGraphModel('/models/model.json')` 即可正確載入。